# Problem 2


In [ ]:
!pip install jax jaxlib matplotlib numpy
import jax
import jax.numpy as jnp
import time
import matplotlib.pyplot as plt





# Part 1

In [ ]:

def elementwise_ops(x):
  x = jnp.sin(x)
  x = jnp.cos(x)
  x = jnp.exp(x)
  x = jnp.square(x)
  x = jnp.sinh(x)
  x = jnp.log(jnp.abs(x) + 1e-6)
  x = jnp.exp(x)
  x = jnp.cos(x)
  x = jnp.tanh(x)
  x = jnp.square(x)
  return x

elementwise_ops_jit = jax.jit(elementwise_ops)

In [ ]:
def time_fn(fn, x, n_warmup=0, n_runs=1):
    """
    Time a JAX function call.

    Parameters
    ----------
    fn      : callable  – function to time
    x       : jnp array – input
    n_warmup: int       – warm-up calls before timing
    n_runs  : int       – number of timed runs (returns min)

    Returns
    -------
    elapsed : float – elapsed time in seconds
    """
    for _ in range(n_warmup):
        fn(x).block_until_ready()

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        result = fn(x).block_until_ready()
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return min(times)  # report best (most stable) run

In [ ]:
matrix_sizes = [100, 500, 1000, 5000] # 10000, 15000, 20000]

results = {
    'eager':      [],
    'jit_first':  [],
    'jit_second': [],
}


print(f"{'Size':>10}  {'Eager (s)':>12}  {'JIT 1st (s)':>13}  {'JIT 2nd (s)':>13}  {'Speedup 2nd/Eager':>18}")
print("-" * 75)

key = jax.random.PRNGKey(42)

for size in matrix_sizes:
    key, subkey = jax.random.split(key)
    x = jax.random.uniform(subkey, shape=(size, size), dtype=jnp.float32)

    t_eager = time_fn(elementwise_ops, x, n_warmup=1, n_runs=15)

    jax.clear_caches()
    fresh_jit_fn = jax.jit(elementwise_ops)
    t_first  = time_fn(fresh_jit_fn, x, n_warmup=0, n_runs=1)
    t_second = time_fn(fresh_jit_fn, x, n_warmup=0, n_runs=15)

    results['eager'].append(t_eager)
    results['jit_first'].append(t_first)
    results['jit_second'].append(t_second)

    speedup = t_eager / t_second if t_second > 0 else float('inf')
    print(f"{size:>4}x{size:<4}  {t_eager:>12.4f}  {t_first:>13.4f}  {t_second:>13.4f}  {speedup:>18.2f}x")

In [ ]:
# ── Styling ──────────────────────────────────────────────────────
import matplotlib.ticker as ticker
plt.rcParams.update({
    'font.family':      'monospace',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

COLORS = {
    'eager':      '#E07B54',
    'jit_first':  '#5B8DB8',
    'jit_second': '#4CAF82',
}

labels = [f"{s}×{s}" for s in matrix_sizes]
x_pos = jnp.arange(len(matrix_sizes))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("JAX Compilation Overhead vs Matrix Size", fontsize=14, fontweight='bold', y=1.01)

# ── Left: absolute times ─────────────────────────────────────────
ax = axes[0]
ax.plot(x_pos, results['eager'],      'o-', color=COLORS['eager'],
        linewidth=2.2, markersize=7, label='Eager (no JIT)')
ax.plot(x_pos, results['jit_first'],  's-', color=COLORS['jit_first'],
        linewidth=2.2, markersize=7, label='JIT – 1st call (compile)')
ax.plot(x_pos, results['jit_second'], '^-', color=COLORS['jit_second'],
        linewidth=2.2, markersize=7, label='JIT – 2nd call (cached)')

ax.set_xticks(x_pos)
ax.set_yscale('log')
ax.set_xticklabels(labels, fontsize=10)
ax.set_xlabel("Matrix Size", fontsize=11)
ax.set_ylabel("Execution Time (seconds)", fontsize=11)
ax.set_title("Absolute Execution Time", fontsize=12)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())
ax.grid(axis='y', linestyle='--', alpha=0.4)

# ── Right: speedup of JIT-2nd vs Eager ───────────────────────────
ax2 = axes[1]
speedups = [e / s for e, s in zip(results['eager'], results['jit_second'])]
overhead = [f / e for f, e in zip(results['jit_first'], results['eager'])]

bars_sp = ax2.bar(x_pos - 0.2, speedups,  width=0.38, color=COLORS['jit_second'],
                  alpha=0.85, label='Speedup: Eager / JIT-2nd')
bars_oh = ax2.bar(x_pos + 0.2, overhead, width=0.38, color=COLORS['jit_first'],
                  alpha=0.85, label='Overhead: JIT-1st / Eager')

for bar in bars_sp:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f"{bar.get_height():.2f}×", ha='center', va='bottom', fontsize=8)
for bar in bars_oh:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f"{bar.get_height():.2f}×", ha='center', va='bottom', fontsize=8)

ax2.axhline(1.0, color='grey', linestyle='--', linewidth=1.2, alpha=0.6)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(labels, fontsize=10)
ax2.set_xlabel("Matrix Size", fontsize=11)
ax2.set_ylabel("Ratio (×)", fontsize=11)
ax2.set_title("Speedup & Overhead Ratios", fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig("jax_compilation_overhead.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to jax_compilation_overhead.png")

# Results
Eager execution stays close to the same amount of time until 5000 x 5000 where it had around an 8x increase in compute time. I believe this is because the CPU cache doesn't have the space to store the matrix as well as the result. For each operation it reads the matrix and writes the result to RAM

The JIT 1st call is mostly flat with variation in computer time being larger with smaller matrices due to system noise. Overall the JIT 1st call took more time than eager execution for each matrix size. I believe the times for the jax first call are relatively similair because the underlying compilation work is the same for each matrix size. JIT first call was also consistently slower than eager execution most likely due to the compilation overhead.

JIT 2nd call was overall the fastest for each run across all sizes. This is because all the operations are fused into one kernel and the computation graph is already compiled and cached leading to the fastest runtimes.

Both eager execution and JIT 2nd call scale with matrix size, but the overhead ratio of compilation cost decreases with size because the time it takes for the first compilation stays the same with matrix size while the eager execution time increases with matrix size.


# Part 2

In [ ]:
trace_log = []   # collect tracing events for later display

@jax.jit
def row_mean(matrix):
    """Compute the mean of each row in a 2-D matrix."""
    return jnp.mean(matrix, axis=1)


In [ ]:
shapes = [(100, 100), (100, 200), (100, 100), (200, 100)]
labels = [
    "Call 1\n(100×100)\nFirst compile",
    "Call 2\n(100×200)\nRetrace",
    "Call 3\n(100×100)\nCache hit",
    "Call 4\n(200×100)\nRetrace",
]

keys = jax.random.split(jax.random.PRNGKey(42), len(shapes))
matrices = [jax.random.normal(k, s) for k, s in zip(keys, shapes)]

# Cache to simulate real-world reuse (call 3 reuses call 1's binary)
shape_cache = {}
times_ms = []
results = []

for i, (mat, shape) in enumerate(zip(matrices, shapes)):
    if shape not in shape_cache:
        # Fresh JIT wrapper = empty cache = forced recompile
        fresh_fn = jax.jit(lambda x: jnp.mean(x, axis=1))
        t0 = time.perf_counter()
        out = fresh_fn(mat).block_until_ready()
        elapsed = (time.perf_counter() - t0) * 1000
        shape_cache[shape] = fresh_fn   # store for reuse
    else:
        # Reuse the already-compiled function for this shape
        cached_fn = shape_cache[shape]
        t0 = time.perf_counter()
        out = cached_fn(mat).block_until_ready()
        elapsed = (time.perf_counter() - t0) * 1000

    times_ms.append(elapsed)
    results.append(out)
    print(f"Call {i+1} | shape={shape} | {elapsed:.3f} ms")

In [ ]:
seen_shapes = set()

for i, (mat, shape) in enumerate(zip(matrices, shapes)):
    jaxpr = jax.make_jaxpr(row_mean)(mat)

    is_new = shape not in seen_shapes
    tag    = "NEW TRACE" if is_new else "CACHED"
    seen_shapes.add(shape)

    print(f"\n{'='*60}")
    print(f"Call {i+1} | shape={shape} | {tag}")
    print(f"{'='*60}")
    print(jaxpr)
    if not is_new:
        print("  → Same abstract shape as a prior call; JIT reuses cached XLA binary.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

retrace_flags = [False, True, False, True]   # which calls trigger a retrace
colors = ["#e74c3c" if r else "#2ecc71" for r in retrace_flags]

bars = ax.bar(range(len(shapes)), times_ms, color=colors,
              edgecolor="white", linewidth=1.2, width=0.55)

# Value labels on top of bars
for bar, t in zip(bars, times_ms):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(times_ms) * 0.01,
            f"{t:.3f} ms",
            ha="center", va="bottom", fontsize=10, fontweight="bold", color="#2c3e50")

# Retrace annotations
for idx, (flag, bar) in enumerate(zip(retrace_flags, bars)):
    if flag:
        ax.annotate("retrace",
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 18), textcoords="offset points",
                    ha="center", fontsize=8.5, color="#c0392b",
                    arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.4))

ax.set_xticks(range(len(shapes)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Avg execution time (ms)", fontsize=11)
ax.set_title("JAX JIT Row-Mean: Execution Time per Call\n"
             "(averaged over 5 repeats after warm-up)", fontsize=13, pad=14)
ax.set_ylim(0, max(times_ms) * 1.35)

green_patch = mpatches.Patch(color="#2ecc71", label="Cache hit (no retrace)")
red_patch   = mpatches.Patch(color="#e74c3c", label="Retrace triggered (new shape)")
ax.legend(handles=[green_patch, red_patch], loc="upper right", fontsize=9)

ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("jit_timing.png", dpi=150)
plt.show()
print("Chart saved to jit_timing.png")

Jax retraces in this case when when the input is a new shape. There is a high upfront performance cost whenever a new shape is passed through the function because the resulting graph must be compiled, but each subsequent function call will be faster because it uses the cached computation graph. Performance issues can happen when the input to the function is called frequently with different shapes. In short there is a high upfront cost because of compilation, but after the result is cached making function calls on matricies of the same shape more efficient.

# Part 3

In [ ]:
import torch
import time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# Part 1
def foo_eager(x):
  total = torch.zeros_like(x)
  sin_x = x
  cos_x = x
  for i in range(100):
    sin_x = torch.sin(sin_x)
    cos_x = torch.cos(cos_x)
    total = total + sin_x + cos_x
  return total
# compiled
x = torch.tensor(1.0, device=device)
@torch.jit.script
def foo_jit(x: torch.Tensor) -> torch.Tensor:
    total = torch.zeros_like(x)
    sin_x = x
    cos_x = x
    for i in range(100):
        sin_x = torch.sin(sin_x)
        cos_x = torch.cos(cos_x)
        total = total + sin_x + cos_x
    return total

# warm-up: compiled only
for _ in range(5):
   foo_jit(x)
if device.type == "cuda":
    torch.cuda.synchronize()
N = 200

start = time.perf_counter()
for _ in range(N):
    foo_eager(x)
if device.type == "cuda":
    torch.cuda.synchronize()   # wait for GPU before stopping clock
eager_avg = (time.perf_counter() - start) / N

start = time.perf_counter()
for _ in range(N):
    foo_jit(x)
if device.type == "cuda":
    torch.cuda.synchronize()
compiled_avg = (time.perf_counter() - start) / N

print(f"Eager    avg: {eager_avg*1e3:.4f} ms")
print(f"Compiled avg: {compiled_avg*1e3:.4f} ms")
print(f"Speedup:      {eager_avg/compiled_avg:.2f}x")

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

activities = [ProfilerActivity.CPU]
if device.type == "cuda":
    activities.append(ProfilerActivity.CUDA)

for _ in range(5):
    foo_jit(x)
if device.type == "cuda":
    torch.cuda.synchronize()

for name, fn in [("eager", foo_eager), ("compiled", foo_jit)]:
    with profile(
        activities=activities,
        profile_memory=True,
        acc_events=True,
    ) as prof:
        with record_function(name):
            for _ in range(200):
                fn(x)
            if device.type == "cuda":
                torch.cuda.synchronize()

    raw_events = [e for e in prof.events() if e.device_time_total > 0]

    num_kernels = len(raw_events) // 200          # per single run
    total_time  = sum(e.device_time_total for e in prof.events()) / 1e3 / 200  # ms per run
    mem         = sum(e.device_memory_usage for e in prof.events()) / 1024     # KB

    print(f"\n{'─'*40}")
    print(f"  Version          : {name}")
    print(f"  Kernels launched : {num_kernels}")
    print(f"  Total time       : {total_time:.3f} ms")
    print(f"  Memory throughput: {mem:.2f} KB")
    print(f"{'─'*40}")

# Results
JIT achieves faster because it only creates 40 kernels instead of 800 meaning it has less launch overhead after warm-up. The total memory through put is also 8x greater with eager execution because each operation creates it's own kernel that reads/writes to memory leading to worse performance. The theoretical speed up. Overall operator fusion increases performance because it has a lower memory throughput and less overhead for kernel launches.